# Room Booking Chatbot — Technology Walkthrough

This notebook explains the key technologies used in the solution and shows them in action with runnable code examples.

**Stack:** Python · LangChain · Groq (Llama 3.3) · SQLite · Streamlit

---
## 1. LLM Provider — Groq

Groq is a free LLM inference API (no credit card required). It supports OpenAI-compatible requests and integrates natively with LangChain via `langchain-groq`.

We use the `llama-3.3-70b-versatile` model, which has strong tool-calling support — meaning it can reliably choose which function to invoke and with what arguments based on natural language input.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

response = llm.invoke("What is the capital of France?")
print(response.content)

---
## 2. Tool Calling with LangChain

Tool calling (also called function calling) lets the LLM decide to invoke a predefined Python function instead of generating a text response. The LLM receives the tool's name and docstring, and when it decides to use it, it outputs a structured JSON payload with the arguments.

LangChain's `@tool` decorator turns any Python function into a tool the LLM can call.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.
    
    Args:
        city: The name of the city.
    """
    # In a real app this would call a weather API
    return f"It is sunny and 22°C in {city}."

# The LLM sees this schema when deciding whether to call the tool:
print(get_weather.name)
print(get_weather.description)

---
## 3. The ReAct Agent Pattern

The solution uses LangChain's **ReAct** (Reason + Act) agent. Each turn the agent:
1. **Thinks** about what to do next
2. **Calls a tool** (Action)
3. **Observes** the result
4. Repeats until it has enough information to give a **Final Answer**

This loop allows the agent to chain multiple tool calls in one conversational turn — for example, checking available rooms before creating a booking.

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import ChatPromptTemplate

# Minimal example prompt for a ReAct agent
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant.\n\nTools available:\n{tools}\n
Use this format:\nQuestion: input\nThought: reasoning\nAction: tool name from [{tool_names}]\nAction Input: tool args\nObservation: result\nFinal Answer: your reply"""),
    ("human", "{input}\n\n{agent_scratchpad}"),
])

tools = [get_weather]
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

result = executor.invoke({"input": "What's the weather like in Buenos Aires?"})
print("\nFinal answer:", result["output"])

---
## 4. Booking Store — SQLite Layer

All booking logic lives in `booking_store.py` and uses SQLite for persistence. The store enforces every business rule deterministically — the LLM cannot create an invalid booking because the tools return structured errors if constraints are violated.

In [ ]:
from src import booking_store

# Initialize a fresh in-memory-style DB for the demo
booking_store.DB_PATH = ":memory:"

# We need to re-import after changing the path — easier to inline the setup here
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE IF NOT EXISTS bookings (
        id TEXT PRIMARY KEY, room TEXT, title TEXT,
        attendee_count INTEGER, start_dt TEXT, end_dt TEXT, owner TEXT
    )
""")
conn.commit()

booking_store.init_db()
print("Rooms and capacities:", booking_store.ROOMS)

In [ ]:
# Create a valid booking
result = booking_store.create_booking(
    room="B",
    title="Team Sync",
    attendees=4,
    start_dt="2026-07-20 10:00",
    end_dt="2026-07-20 11:00",
    owner="User1",
)
print("Created:", result)

In [ ]:
# Try to double-book the same room — should fail
result = booking_store.create_booking(
    room="B",
    title="Another Meeting",
    attendees=2,
    start_dt="2026-07-20 10:30",
    end_dt="2026-07-20 11:30",
    owner="User2",
)
print("Overlap attempt:", result)

In [ ]:
# List available rooms for a time range
available = booking_store.list_available_rooms(
    start_dt="2026-07-20 10:00",
    end_dt="2026-07-20 11:00",
    attendees=4,
)
print("Available rooms (10:00–11:00, 4 people):", available)

In [ ]:
# View room B's schedule for the day
schedule = booking_store.get_room_schedule("B", "2026-07-20")
# Print just the occupied/non-available slots for brevity
occupied = [s for s in schedule["slots"] if s["status"] == "occupied"]
print(f"Occupied slots in Room B on 2026-07-20: {occupied}")

In [ ]:
# List User1's bookings
my_bookings = booking_store.list_my_bookings("User1")
print("User1 bookings:", my_bookings)

---
## 5. Tool Binding per User

Each Streamlit session creates a fresh set of tools via `make_tools(current_user)`. The `current_user` is captured in a closure, so the LLM never sees or controls which user is performing the action — preventing identity spoofing.

In [ ]:
from src.tools import make_tools

user1_tools = make_tools("User1")
print("Tools available to User1:")
for t in user1_tools:
    print(f"  {t.name}: {t.description[:60]}...")

---
## 6. End-to-End Agent Interaction

Here is what a real conversation looks like when the agent is wired up with Groq and the booking tools. (Requires a valid `GROQ_API_KEY` in `.env`.)

In [ ]:
from src.agent import build_agent_executor, run_agent

executor = build_agent_executor("User1")
history = []

questions = [
    "What rooms are available tomorrow at 2pm for 5 people for 1 hour?",
    "Book Room C for a Product Review meeting with 5 attendees from 14:00 to 15:00 on 2026-07-17.",
    "Show me my bookings.",
]

for q in questions:
    print(f"\nUser: {q}")
    answer = run_agent(executor, q, history, "User1")
    print(f"Assistant: {answer}")
    history.append({"role": "user", "content": q})
    history.append({"role": "assistant", "content": answer})